# Rocket Roll Dynamics — Physics, State-Space Model & Control Design
Based on [RocketPy Roll Equations](https://docs.rocketpy.org/en/v1.0.0/technical/aerodynamics/roll_equations.html)

---

## 1. What's Actually Happening

Roll is a **tug-of-war** between two aerodynamic moments that both scale with dynamic pressure $\bar{q} = \frac{1}{2}\rho v_0^2$:

| Moment | Driver | Scales with |
|--------|--------|-------------|
| **Forcing** $M_f$ | Fin cant angle $\delta$ — fins act like a propeller | $\bar{q}$, $\delta$ |
| **Damping** $M_d$ | Rotation itself — spinning fins see an extra velocity component $\omega\xi$, producing a restoring force | $\bar{q}$, $\omega$ |

Because both scale with $\bar{q}$, **they grow and shrink together with velocity**. This means $\bar{q}$ cancels out of the steady-state expression, giving a clean result:

$$\omega_{ss} = \frac{C_{lf\delta}}{C_{ld}} \cdot \frac{2v_0\,\delta}{L_r}$$

**Key properties of steady-state:**
- Linear in $\delta$ — double the cant, double the roll rate
- Linear in $v_0$ — go twice as fast, spin twice as fast  
- Independent of air density — altitude doesn't matter at steady state

---

## 2. Equations of Motion

$$M_{roll} = M_f - M_d$$

$$M_f = N\,(Y_{MA}+r_t)\,(C_{N\alpha})_1\,\delta\,\bar{q}\,A_r \equiv C_{lf\delta}\,\delta\,\bar{q}\,A_r\,L_r$$

$$M_d = C_{ld}\,\frac{\omega L_r}{2v_0}\,\bar{q}\,A_r\,L_r$$

$$\boxed{I_{xx}\,\dot{\omega} = C_{lf\delta}\,\delta\,\bar{q}\,A_r\,L_r - C_{ld}\,\frac{\omega L_r}{2v_0}\,\bar{q}\,A_r\,L_r}$$

For trapezoidal fins, $C_{ld}$ requires:
$$C_{ld} = \frac{2N\,(C_{N\alpha})_1\cos\delta}{A_r L_r^2}\int_{r_t}^{s+r_t}c(\xi)\xi^2\,d\xi$$
$$\int_{r_t}^{s+r_t}c(\xi)\xi^2\,d\xi = \frac{s}{12}\left[(C_r+3C_t)s^2+4(C_r+2C_t)sr_t+6(C_r+C_t)r_t^2\right]$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.linalg import solve_continuous_are
from scipy.signal import bode, lti

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 3. Rocket & Fin Parameters

In [ ]:
# ── Rocket geometry ───────────────────────────────────────────────────────────
N    = 4            # number of fins
r_t  = 0.038        # tube radius at fin root [m]  (76 mm diameter)
s    = 0.080        # fin semi-span [m]
C_r  = 0.120        # root chord [m]
C_t  = 0.060        # tip chord [m]

L_r  = 2 * r_t            # reference length = diameter [m]
A_r  = np.pi * r_t**2     # reference area [m²]

# ── Fin aerodynamics (Barrowman) ──────────────────────────────────────────────
AR_fin = 2 * s**2 / (0.5 * (C_r + C_t) * s)
CNa1   = 2 * np.pi * AR_fin / (2 + np.sqrt(4 + AR_fin**2))   # per fin, per rad
Y_MA   = s / 3 * (C_r + 2*C_t) / (C_r + C_t)                # MAC span location [m]

print(f"Fin aspect ratio : {AR_fin:.3f}")
print(f"(CNα)₁           : {CNa1:.4f} /rad")
print(f"Y_MA             : {Y_MA*1e3:.2f} mm from fin root")

# ── Inertia & actuation ───────────────────────────────────────────────────────
I_xx      = 0.015   # roll moment of inertia [kg·m²]
delta_deg = 2.0     # fin cant angle [degrees]  — the control input
delta     = np.radians(delta_deg)

# ── Atmosphere ────────────────────────────────────────────────────────────────
rho = 1.225         # air density [kg/m³]

## 4. Aerodynamic Coefficients

In [ ]:
def fin_integral_trap(C_r, C_t, s, r_t):
    """Closed-form ∫c(ξ)ξ² dξ from r_t to s+r_t for trapezoidal fins."""
    return s / 12 * (
        (C_r + 3*C_t) * s**2
        + 4*(C_r + 2*C_t) * s * r_t
        + 6*(C_r +   C_t) * r_t**2
    )

I_fin = fin_integral_trap(C_r, C_t, s, r_t)

# Roll forcing coefficient slope  (M_f = Clf_delta * δ * q̄ * Ar * Lr)
Clf_delta = N * (Y_MA + r_t) * CNa1 / L_r

# Roll damping coefficient  (M_d = Cld * (ωLr/2v0) * q̄ * Ar * Lr)
Cld = (2 * N * CNa1 * np.cos(delta)) / (A_r * L_r**2) * I_fin

print(f"Fin integral  : {I_fin:.6f} m⁴")
print(f"Clf_delta     : {Clf_delta:.4f} /rad")
print(f"Cld           : {Cld:.4f} /rad")

## 5. Steady-State Roll Rate

Setting $\dot\omega = 0$ the $\bar{q}$ terms cancel, giving a linear relationship with both $v_0$ and $\delta$.

In [ ]:
v0_range = np.linspace(10, 300, 500)
omega_ss = (Clf_delta * delta) / Cld * (2 * v0_range / L_r)
rpm_ss   = omega_ss * 60 / (2 * np.pi)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(v0_range, omega_ss, 'steelblue')
axes[0].set(xlabel='Speed $v_0$ [m/s]', ylabel='$\\omega_{ss}$ [rad/s]',
            title=f'Steady-State Roll Rate  (δ = {delta_deg}°)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(v0_range, rpm_ss, 'tomato')
axes[1].set(xlabel='Speed $v_0$ [m/s]', ylabel='$\\omega_{ss}$ [RPM]',
            title=f'Steady-State Roll Rate  (δ = {delta_deg}°)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

v0_ex = 150.0
oss = (Clf_delta * delta) / Cld * (2 * v0_ex / L_r)
print(f"At v0 = {v0_ex} m/s, δ = {delta_deg}°  →  ω_ss = {oss:.2f} rad/s = {oss*60/(2*np.pi):.1f} RPM")

---

## 6. Linearized State-Space Model

The EOM is already linear in $\omega$ and $\delta$. Divide through by $I_{xx}$ and collect terms:

$$\dot{\omega} = \underbrace{-\frac{C_{ld}\,\bar{q}\,A_r\,L_r^2}{2v_0\,I_{xx}}}_{-a(v_0)}\,\omega + \underbrace{\frac{C_{lf\delta}\,\bar{q}\,A_r\,L_r}{I_{xx}}}_{b(v_0)}\,\delta$$

Both gains are **functions of $v_0$** (and hence time), making this a **linear time-varying (LTV)** system:

| Gain | Expression | Scales with |
|------|-----------|-------------|
| Damping $a(v_0)$ | $\frac{C_{ld}\,\bar{q}\,A_r\,L_r^2}{2v_0\,I_{xx}}$ | $v_0$ (∝ $\rho v_0^2 / v_0$) |
| Input $b(v_0)$ | $\frac{C_{lf\delta}\,\bar{q}\,A_r\,L_r}{I_{xx}}$ | $v_0^2$ |

### Two operating-point choices

**Roll-rate control** (1-state):  
State $x = \omega$, input $u = \delta$
$$\dot x = -a\,x + b\,u, \qquad y = x$$

**Roll-angle control** (2-state):  
State $\mathbf{x} = [\omega,\, \phi]^T$, input $u = \delta$
$$\dot{\mathbf{x}} = \underbrace{\begin{bmatrix}-a & 0 \\ 1 & 0\end{bmatrix}}_{A}\mathbf{x} + \underbrace{\begin{bmatrix}b \\ 0\end{bmatrix}}_{B}u$$

> Note: the 2-state system has a pole at $s=0$ (integrator from $\omega\to\phi$), so a pure P controller on $\phi$ will have zero steady-state error, but LQR is better suited here.

### Time constant

$$\tau(v_0) = \frac{1}{a(v_0)} = \frac{2v_0\,I_{xx}}{C_{ld}\,\bar{q}\,A_r\,L_r^2}$$

This **shrinks with increasing $v_0$** — the rocket responds faster at max-$q$, and is sluggish at low speed.

In [ ]:
def ss_gains(v0, rho=1.225):
    """Return (a, b) — the LTV state-space gains at a given speed."""
    q_bar = 0.5 * rho * v0**2
    a = (Cld    * q_bar * A_r * L_r**2) / (2 * v0 * I_xx)
    b = (Clf_delta * q_bar * A_r * L_r) / I_xx
    return a, b

# ── Evaluate over flight ──────────────────────────────────────────────────────
v0_arr = np.linspace(10, 300, 500)
a_arr  = np.array([ss_gains(v)[0] for v in v0_arr])
b_arr  = np.array([ss_gains(v)[1] for v in v0_arr])
tau_arr = 1.0 / a_arr
Kdc_arr = b_arr / a_arr   # DC gain  ω_ss / δ

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

axes[0,0].plot(v0_arr, a_arr, 'steelblue')
axes[0,0].set(xlabel='$v_0$ [m/s]', ylabel='$a$ [1/s]', title='Damping gain $a(v_0)$')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(v0_arr, b_arr, 'tomato')
axes[0,1].set(xlabel='$v_0$ [m/s]', ylabel='$b$ [rad/s per rad]', title='Input gain $b(v_0)$')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(v0_arr, tau_arr, 'mediumpurple')
axes[1,0].set(xlabel='$v_0$ [m/s]', ylabel='$\\tau$ [s]', title='Time constant $\\tau = 1/a$')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(v0_arr, Kdc_arr * (180/np.pi), 'darkorange')   # rad/s per degree
axes[1,1].set(xlabel='$v_0$ [m/s]', ylabel='$K_{dc}$  [(rad/s)/deg]',
              title='DC gain  $\\omega_{ss}/\\delta$')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('State-Space Gains vs. Flight Speed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print table at a few key speeds
print(f"{'v0 [m/s]':>10} {'a [1/s]':>10} {'b [1/s]':>10} {'τ [s]':>8} {'Kdc [(rad/s)/deg]':>20}")
print('-'*65)
for v0 in [50, 100, 150, 200, 250]:
    a, b = ss_gains(v0)
    print(f"{v0:>10.0f} {a:>10.3f} {b:>10.2f} {1/a:>8.4f} {b/a*(np.pi/180):>20.3f}")

## 7. Transfer Function & Bode Plot (at Operating Point)

At a fixed $v_0$, the roll-rate plant is first-order:

$$G(s) = \frac{\omega(s)}{\delta(s)} = \frac{b}{s + a}$$

DC gain $K_{dc} = b/a$, pole at $s = -a$.

In [ ]:
op_speeds = [100, 150, 200]   # operating points [m/s]

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
w_range = np.logspace(-1, 3, 500)

for v0 in op_speeds:
    a, b = ss_gains(v0)
    sys = lti([b], [1, a])
    w, mag, phase = bode(sys, w=w_range)
    axes[0].semilogx(w, mag,   label=f'$v_0$ = {v0} m/s')
    axes[1].semilogx(w, phase, label=f'$v_0$ = {v0} m/s')

axes[0].set(ylabel='Magnitude [dB]', title='Bode Plot: Roll-Rate Plant $G(s) = b/(s+a)$')
axes[0].legend(); axes[0].grid(True, which='both', alpha=0.3)
axes[1].set(xlabel='Frequency [rad/s]', ylabel='Phase [deg]')
axes[1].legend(); axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# Open-loop step response (input = 1 rad cant, output = rad/s roll rate)
fig, ax = plt.subplots(figsize=(10, 4))
t_step = np.linspace(0, 1.0, 1000)
for v0 in op_speeds:
    a, b = ss_gains(v0)
    # Step response of b/(s+a): ω(t) = (b/a)(1 - e^{-at})
    omega_step = (b/a) * (1 - np.exp(-a * t_step))
    ax.plot(t_step, np.degrees(omega_step), label=f'$v_0$ = {v0} m/s')

ax.set(xlabel='Time [s]', ylabel='Roll rate [deg/s]',
       title='Open-Loop Step Response to δ = 1 rad (57°) cant — for scale only')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 8. Control Design

### 8a. Choosing your control objective

| Objective | States | Use case |
|-----------|--------|----------|
| **Track $\omega_{ref}$** (roll rate) | $x = \omega$ | Spin stabilisation at a target RPM |
| **Track $\phi_{ref}$** (roll angle) | $x = [\omega, \phi]$ | Azimuth lock, camera pointing |

### 8b. PID — Roll Rate Control

Plant: $G(s) = b/(s+a)$. With proportional feedback $\delta = K_p(\omega_{ref}-\omega)$:

$$G_{cl}(s) = \frac{K_p b}{s + a + K_p b}$$

- Closed-loop time constant: $\tau_{cl} = 1/(a + K_p b)$
- There **is** steady-state error because the open-loop has no integrator: $e_{ss} = \frac{a}{a + K_p b}$
- **PI** eliminates this: $G_c(s) = K_p\left(1 + \frac{1}{T_i s}\right)$

### 8c. LQR — Roll Rate Control (1-state)

Scalar system: $\dot\omega = -a\omega + b\delta$. Minimise:
$$J = \int_0^\infty Q\,\omega^2 + R\,\delta^2\,dt$$

The ARE reduces to a quadratic; optimal gain: $K^* = \frac{1}{R}\,B\,P = \frac{b\,P}{R}$

### 8d. LQR — Roll Angle Control (2-state)

$$A = \begin{bmatrix}-a & 0\\ 1 & 0\end{bmatrix}, \quad B = \begin{bmatrix}b\\0\end{bmatrix}, \quad Q = \begin{bmatrix}q_\omega & 0\\ 0 & q_\phi\end{bmatrix}, \quad R = r_\delta$$

Solve the continuous-time ARE: $A^TP + PA - PBR^{-1}B^TP + Q = 0$ → $K = R^{-1}B^TP$

In [ ]:
# ── Operating point for controller design ────────────────────────────────────
v0_design = 150.0   # [m/s]  — design at max-q or midpoint of interest
a0, b0 = ss_gains(v0_design)
print(f"Design point: v0 = {v0_design} m/s")
print(f"  a = {a0:.4f} 1/s   (open-loop pole at s = -{a0:.4f})")
print(f"  b = {b0:.4f} 1/s   (input gain)")
print(f"  τ = {1/a0:.4f} s")
print(f"  K_dc = {b0/a0:.4f} rad/s per rad cant = {b0/a0*np.pi/180:.4f} rad/s per deg cant")

In [ ]:
# ── 8b. PID — Roll Rate ───────────────────────────────────────────────────────
# Desired closed-loop time constant
tau_cl_desired = 0.05    # [s]  — 5× faster than open loop
omega_cl_desired = 1.0 / tau_cl_desired

# P gain: τ_cl = 1/(a + Kp*b) → Kp = (1/τ_cl - a) / b
Kp_rate = (omega_cl_desired - a0) / b0

# PI — integral time constant (place integrator at ~10% of closed-loop BW)
Ti = 10 * tau_cl_desired   # integral time [s]
Kp_pi = Kp_rate            # same proportional gain
Ki_pi = Kp_pi / Ti

print(f"PID Roll Rate Controller (design point v0 = {v0_design} m/s)")
print(f"  Target τ_cl = {tau_cl_desired} s")
print(f"  P only : Kp = {Kp_rate:.4f} rad_cant / (rad/s)")
print(f"  PI     : Kp = {Kp_pi:.4f},  Ki = {Ki_pi:.4f}")

In [ ]:
# ── 8c. LQR — Roll Rate (1-state) ────────────────────────────────────────────
# Tune Q, R: higher Q/R penalises state error more (aggressive), lower = more cautious
Q_rate = np.array([[1.0]])    # weight on ω error  [1/(rad/s)²]
R_rate = np.array([[0.01]])   # weight on δ         [1/rad²]

A1 = np.array([[-a0]])
B1 = np.array([[b0]])

P1 = solve_continuous_are(A1, B1, Q_rate, R_rate)
K_lqr_rate = (1/R_rate[0,0]) * B1.T @ P1
cl_pole_lqr = A1[0,0] - B1[0,0] * K_lqr_rate[0,0]

print(f"LQR Roll-Rate Controller (Q={Q_rate[0,0]}, R={R_rate[0,0]})")
print(f"  K_lqr     = {K_lqr_rate[0,0]:.4f} rad_cant / (rad/s)")
print(f"  CL pole   = {cl_pole_lqr:.4f}   (τ_cl = {-1/cl_pole_lqr:.4f} s)")
print(f"  Gain margin: compare with Kp_P = {Kp_rate:.4f}")

In [ ]:
# ── 8d. LQR — Roll Angle (2-state) ───────────────────────────────────────────
# State: [ω, φ],  input: δ
A2 = np.array([[-a0, 0.0],
               [ 1.0, 0.0]])
B2 = np.array([[b0],
               [0.0]])

# Q weights: penalise angle error more than rate error
q_omega = 1.0     # weight on ω  [1/(rad/s)²]
q_phi   = 10.0    # weight on φ  [1/rad²]   — higher = tighter angle control
r_delta = 0.1     # weight on δ  [1/rad²]

Q2 = np.diag([q_omega, q_phi])
R2 = np.array([[r_delta]])

P2 = solve_continuous_are(A2, B2, Q2, R2)
K_lqr_angle = (1/R2[0,0]) * B2.T @ P2
K_omega, K_phi = K_lqr_angle[0]

cl_A2 = A2 - B2 @ K_lqr_angle
cl_poles = np.linalg.eigvals(cl_A2)

print(f"LQR Roll-Angle Controller (q_ω={q_omega}, q_φ={q_phi}, r_δ={r_delta})")
print(f"  K_ω  = {K_omega:.4f}  (rate feedback)")
print(f"  K_φ  = {K_phi:.4f}  (angle feedback)")
print(f"  δ = -K_ω·ω - K_φ·φ  +  feed-forward")
print(f"  Closed-loop poles: {cl_poles}")

## 9. Closed-Loop Step Response Comparison

In [ ]:
t_cl = np.linspace(0, 0.5, 2000)
omega_ref = np.radians(180)   # target: 180 deg/s = π rad/s

# ── Simulate controllers at the design operating point ────────────────────────
def simulate_controller(t_arr, v0_fn, controller_fn, x0=[0.0, 0.0], method='RK45'):
    """Integrate roll dynamics with a feedback controller.
    controller_fn(omega, phi, omega_ref) -> delta [rad]"""
    def ode(t, y):
        omega, phi = y
        v0  = v0_fn(t)
        a_, b_ = ss_gains(v0)
        delta_cmd = np.clip(controller_fn(omega, phi), -np.radians(10), np.radians(10))
        domega = -a_ * omega + b_ * delta_cmd
        dphi   = omega
        return [domega, dphi]
    sol = solve_ivp(ode, (t_arr[0], t_arr[-1]), x0, t_eval=t_arr, method=method, max_step=1e-3)
    return sol.y[0], sol.y[1]

v0_const = lambda t: v0_design   # freeze at design point for comparison

# P controller
ctrl_P  = lambda w, ph: Kp_rate * (omega_ref - w)
# PI controller (approximate with Euler-integrated error externally)
# For simplicity embed integral in ODE as augmented state
def simulate_PI(t_arr, v0_fn, Kp, Ki, omega_ref):
    def ode(t, y):
        omega, phi, e_int = y
        v0  = v0_fn(t)
        a_, b_ = ss_gains(v0)
        err   = omega_ref - omega
        delta_cmd = np.clip(Kp * err + Ki * e_int, -np.radians(10), np.radians(10))
        domega = -a_ * omega + b_ * delta_cmd
        return [domega, omega, err]
    sol = solve_ivp(ode, (t_arr[0], t_arr[-1]), [0.0, 0.0, 0.0], t_eval=t_arr, max_step=1e-3)
    return sol.y[0], sol.y[1]

# LQR rate
ctrl_LQR_rate = lambda w, ph: -K_lqr_rate[0,0] * (w - omega_ref)   # error form

omega_P,   phi_P   = simulate_controller(t_cl, v0_const, ctrl_P)
omega_PI,  phi_PI  = simulate_PI(t_cl, v0_const, Kp_pi, Ki_pi, omega_ref)
omega_LQR, phi_LQR = simulate_controller(t_cl, v0_const, ctrl_LQR_rate)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ref_line = np.degrees(omega_ref) * np.ones_like(t_cl)
axes[0].plot(t_cl, np.degrees(omega_P),   label='P only')
axes[0].plot(t_cl, np.degrees(omega_PI),  label='PI')
axes[0].plot(t_cl, np.degrees(omega_LQR), label='LQR (rate)')
axes[0].plot(t_cl, ref_line, 'k--', alpha=0.5, label='Reference')
axes[0].set(xlabel='Time [s]', ylabel='Roll rate [deg/s]',
            title=f'Roll Rate Step Response  (v0 = {v0_design} m/s)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_cl, np.degrees(phi_P),   label='P only')
axes[1].plot(t_cl, np.degrees(phi_PI),  label='PI')
axes[1].plot(t_cl, np.degrees(phi_LQR), label='LQR (rate)')
axes[1].set(xlabel='Time [s]', ylabel='Roll angle [deg]',
            title='Roll Angle (integrating ω)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

ss_P   = np.degrees(omega_P[-1])
ss_PI  = np.degrees(omega_PI[-1])
ss_LQR = np.degrees(omega_LQR[-1])
ref_d  = np.degrees(omega_ref)
print(f"Steady-state error (ref = {ref_d:.1f} deg/s):")
print(f"  P   : {ss_P:.2f} deg/s  (error = {ref_d - ss_P:.2f} deg/s = {100*(ref_d-ss_P)/ref_d:.1f}%)")
print(f"  PI  : {ss_PI:.2f} deg/s  (error = {ref_d - ss_PI:.2f} deg/s)")
print(f"  LQR : {ss_LQR:.2f} deg/s  (error = {ref_d - ss_LQR:.2f} deg/s)")

## 10. Gain Scheduling Over the Flight Profile

Because $a(v_0)$ and $b(v_0)$ vary significantly with speed, a fixed-gain controller designed at one operating point will be **overly aggressive at high speed** and **sluggish at low speed**. Gain scheduling computes $K_p(v_0)$ to maintain a constant closed-loop time constant $\tau_{cl}^*$ throughout the flight.

In [ ]:
# ── Velocity profile ──────────────────────────────────────────────────────────
t_burn = 2.5
v_max  = 200.0

def v0_profile(t):
    if t <= t_burn:
        return max(5.0, v_max * t / t_burn)
    return v_max * np.exp(-0.05 * (t - t_burn))

v0_vec = np.vectorize(v0_profile)

# ── Scheduled gain: Kp(v0) = (1/τ*_cl - a) / b ───────────────────────────────
tau_cl_target = 0.08   # [s]  target closed-loop time constant

v0_sched = np.linspace(10, 300, 400)
Kp_sched = np.array([(1/tau_cl_target - ss_gains(v)[0]) / ss_gains(v)[1]
                     for v in v0_sched])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(v0_sched, Kp_sched, 'steelblue', label='Scheduled $K_p(v_0)$')
ax.axhline(Kp_rate, color='tomato', ls='--', label=f'Fixed $K_p$ (designed @ {v0_design} m/s)')
ax.set(xlabel='Speed $v_0$ [m/s]', ylabel='$K_p$ [rad/rad/s]',
       title=f'Gain Schedule for $\\tau_{{cl}}^* = {tau_cl_target}$ s')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Nonlinear simulation: fixed vs. scheduled gain ────────────────────────────
from scipy.interpolate import interp1d

Kp_interp = interp1d(v0_sched, Kp_sched, fill_value='extrapolate')

t_sim = np.arange(0, 10.0, 0.005)

def roll_ode_ctrl(t, y, Kp_fn, omega_ref_val):
    omega, phi = y
    v0    = v0_profile(t)
    q_bar = 0.5 * rho * v0**2
    kp    = float(Kp_fn(v0))
    delta_cmd = np.clip(kp * (omega_ref_val - omega), -np.radians(10), np.radians(10))
    a_, b_ = ss_gains(v0)
    return [-a_*omega + b_*delta_cmd, omega]

omega_ref_sim = np.radians(180)   # 180 deg/s target

sol_fixed  = solve_ivp(roll_ode_ctrl, (0, 10), [0,0], t_eval=t_sim, max_step=5e-3,
                       args=(lambda v: Kp_rate, omega_ref_sim))
sol_sched  = solve_ivp(roll_ode_ctrl, (0, 10), [0,0], t_eval=t_sim, max_step=5e-3,
                       args=(Kp_interp, omega_ref_sim))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(sol_fixed.t, v0_vec(sol_fixed.t), 'k', alpha=0.7)
axes[0].axvline(t_burn, color='gray', ls='--', alpha=0.5, label='Burnout')
axes[0].set(ylabel='Speed [m/s]', title='Flight Velocity Profile')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

ref_arr = np.degrees(omega_ref_sim) * np.ones_like(sol_fixed.t)
axes[1].plot(sol_fixed.t, np.degrees(sol_fixed.y[0]), label='Fixed $K_p$', color='tomato')
axes[1].plot(sol_sched.t, np.degrees(sol_sched.y[0]), label='Scheduled $K_p(v_0)$', color='steelblue')
axes[1].plot(sol_fixed.t, ref_arr, 'k--', alpha=0.5, label='Reference')
axes[1].axvline(t_burn, color='gray', ls='--', alpha=0.5)
axes[1].set(xlabel='Time [s]', ylabel='Roll rate [deg/s]',
            title='Closed-Loop Roll Rate: Fixed vs. Gain-Scheduled P Controller')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Parametric Study: Cant Angle Effect (Open Loop)

In [ ]:
cant_angles = [0.5, 1.0, 2.0, 3.0, 5.0]
t_ol = np.arange(0, 10.0, 0.005)

fig, ax = plt.subplots(figsize=(10, 5))
for d_deg in cant_angles:
    d = np.radians(d_deg)
    Cld_d = (2 * N * CNa1 * np.cos(d)) / (A_r * L_r**2) * I_fin

    def ode_ol(t, y, d=d, Cld_d=Cld_d):
        omega, phi = y
        v0    = v0_profile(t)
        q_bar = 0.5 * rho * v0**2
        Mf = Clf_delta * d * q_bar * A_r * L_r
        Md = Cld_d * (omega * L_r / (2 * v0 + 1e-9)) * q_bar * A_r * L_r
        return [(Mf - Md) / I_xx, omega]

    sol_ol = solve_ivp(ode_ol, (0, 10), [0,0], t_eval=t_ol, max_step=5e-3)
    ax.plot(sol_ol.t, sol_ol.y[0]*60/(2*np.pi), label=f'δ = {d_deg}°')

ax.axvline(t_burn, color='gray', ls='--', label='Burnout')
ax.set(xlabel='Time [s]', ylabel='Roll rate [RPM]', title='Open-Loop Roll Rate vs. Cant Angle')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 12. Design Summary

| Controller | Steady-state error | Best for |
|------------|-------------------|----------|
| **P** | Yes (depends on $a/b$) | Quick prototype, insensitive to noise |
| **PI** | No | Roll-rate tracking with zero SS error |
| **LQR (1-state)** | No (integral action via error-state) | Optimal rate control with actuator penalty |
| **LQR (2-state)** | No | Full roll-angle lock |
| **Gain-scheduled** | Any of above | Consistent performance across Mach |

**Critical unknowns that affect your gains:**
- $I_{xx}$ — scales $\tau$ directly; weigh or compute from CAD
- $(C_{N\alpha})_1$ at actual Reynolds number — consider a CFD/wind-tunnel correction factor
- Structural twist under fin aerodynamic load — effective $\delta$ ≠ commanded $\delta$ at high $q$
- Actuator delay / bandwidth — add a first-order actuator model $1/(\tau_{act}s+1)$ before designing gains if using a servo

**Next steps for a real design:**
1. Measure / compute $I_{xx}$ accurately
2. Add actuator model (servo bandwidth ~50–100 Hz typical)
3. Run Monte Carlo over $\pm20\%$ uncertainty in $I_{xx}$ and $C_{N\alpha}$
4. Verify closed-loop stability margins with Bode/Nyquist